# Laboratorio 2: 

## Ejercicio 1

### Razonamiento previo (AES, CBC)

- AES (Advanced Encryption Standard): Es un algoritmo de cifrado por bloques. Esto significa que no cifra el mensaje letra por letra, sino que lo corta en bloques fijos, en este caso de 16 bytes (128 bits).   

- Modo CBC (Cipher Block Chaining): Es una regla de operación para conectar los bloques que corta AES. Su lógica dicta que, antes de cifrar un bloque de texto plano ($P_i$), se debe mezclar mediante una operación XOR ($\oplus$) con el bloque de texto cifrado anterior ($C_{i-1}$).   

    - La fórmula criptográfica es: $C_i = E_k(P_i \oplus C_{i-1})$
    
- Vector de Inicialización (IV): Si aplicamos la fórmula anterior al primer bloque del mensaje ($P_1$), tenemos un problema: no existe un bloque cifrado anterior ($C_0$). Para resolverlo, se inventa un bloque inicial artificial llamado IV. La regla de oro en criptografía es que el IV debe ser único y aleatorio para cada mensaje.   

- Padding (Relleno): Si el mensaje tiene por ejemplo 35 bytes, se tienen dos bloques completos de 16 bytes (32 bytes) y sobran 3 bytes. Como AES exige que todos los bloques midan exactamente 16 bytes, se deben rellenar los espacios faltantes. En criptografía real se usan estándares como PKCS7, pero el código de la guía usa un relleno manual sumando caracteres.

In [3]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

key = b'1234567890123456'
iv = b'0000000000000000'

def encrypt(msg):
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    encryptor = cipher.encryptor()
    # Relleno manual sumando asteriscos
    padded = msg + b"*" * (16 - len(msg) % 16)
    return encryptor.update(padded) + encryptor.finalize()

def decrypt(ct):
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    decryptor = cipher.decryptor()
    return decryptor.update(ct) + decryptor.finalize()

msg1 = b"Transfer 100 euros from Ana to Juan"
msg2 = b"Transfer 100 euros from Ana to Luis"

c1 = encrypt(msg1)
c2 = encrypt(msg2)

print("Texto cifrado 1 (hex):", c1.hex())
print("Texto cifrado 2 (hex):", c2.hex())

Texto cifrado 1 (hex): 007948a508cb3f3c77069b214dc6fb6a26bc6fab50605e364b0278de2d874ef847ad3f9617e6435268a91681c18641e9
Texto cifrado 2 (hex): 007948a508cb3f3c77069b214dc6fb6af9a8195d9da7f7d3f5298fd561e4f4213131e406421571006188e4240ab54f9d


a) Ejecute el codigo varias veces y observe los resultados:

El cifrado es el mismo en cada ejecución. Esto se debe a que la key y el iv están fijos en el código, en vez de generarse aleatoriamente.

b) Compare el primer bloque de c1 y c2:

El primer bloque vendria siendo los primeros 32 caracteres. Esto se debe a que ambos mensajes empiezan con "Transfer 100 euros from". Como se aplica la misma formula con la misma clave y iv, el resultado sera obligatoriamente el mismo. Esto es un error grave ya que el atacante puede deducir que ambos mensajes empiezan igual, con las mismas palabras.

c) Identifique 3 problemas de seguridad:
1. IV fijo: esta definido como ceros, no cunmple con que debe ser unico y aleatorio para cada mensaje.
2. Padding inseguro: Al ser asteriscos, si el mensaje terminara con un asterisco, se borraria parte del mensaje.
3. No hay integridad ni autenticación, solo confidencialidad (duh, arreglar esta)

## Ejercicio 2

###  (ECB Electronic Codebook)

A diferencia del modo CBC, el modo ECB no utiliza un Vector de Inicialización (IV) ni encadena los bloques. Su regla de operación dicta que cifra cada bloque de 16 bytes de forma completamente independiente, utilizando únicamente la clave secreta.   

La fórmula criptográfica se reduce a: $C_i = E_k(P_i)$. Esto significa que el texto cifrado del bloque actual ($C_i$) es simplemente el resultado de encriptar el texto plano actual ($P_i$) con la clave ($k$).

In [4]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

key = b'1234567890123456'

def encrypt_ecb(msg):
    cipher = Cipher(algorithms.AES(key), modes.ECB())
    encryptor = cipher.encryptor()
    
    if len(msg) % 16 != 0: 
        raise ValueError("El mensaje debe ocupar bloques completos")
    
    return encryptor.update(msg) + encryptor.finalize()

# Mensaje añadido para demostrar la vulnerabilidad (2 bloques de 16 bytes idénticos)
msg_ecb = b"BLOQUEIDENTICO!!BLOQUEIDENTICO!!" 
c_ecb = encrypt_ecb(msg_ecb)

print("Texto cifrado ECB (hex):", c_ecb.hex())

Texto cifrado ECB (hex): 22277673b4d47cb33a7b84188c40b56e22277673b4d47cb33a7b84188c40b56e


a) Compruebe si los dos bloques de texto cifrado son iguales. ¿Qué propiedad de ECB explica el resultado?

Como ambos bloques son identicos, el cifrado igual lo es. La propiedad que lo explica es el determinismo sin estado de ECB, ya que al aplicar la formula, bloques identicos con la misma clave van a generar cifrados identicos obligatorioamente. No hay un IV previo que altere la matematica del bloque actual.

b) Explique por qué esta propiedad puede filtrar estructura o patrones del mensaje y por qué se considera un problema grave

No oculta la estructura semantica de los datos originales. Si s cifra el archivo de una imagen utilizando ECB, las areas que contengan el mismo color van a generar los mismos bloques cifrados. Un atacante aunque no tenga la clave va a poder deducir visualmente la silueta de la imagen original, o la estructura de una rchivo de texto. rompiendo la confidencialidad.

## Ejercicio 3

### Razonamiento (Fernet)

Hasta ahora hemos trabajado con primitivas de bajo nivel (AES directo en modos CBC y ECB). Cuando trabajas a este nivel, la responsabilidad de hacerlo de forma segura (generar IVs aleatorios, aplicar padding correcto, garantizar que el mensaje no sea alterado) recae completamente en el programador. Como vimos en la Parte 1, es muy fácil equivocarse.   

Aquí es donde entra Fernet. Fernet no es un algoritmo de cifrado en sí, sino una "receta" o construcción criptográfica de alto nivel de la librería cryptography. La teoría detrás de Fernet es que agrupa varias primitivas criptográficas correctas por defecto para que sea imposible para el programador cometer errores básicos de implementación.   

Fernet internamente hace esto:
- Cifrado: Usa AES de 128 bits en modo CBC.
- Padding: Aplica relleno estándar PKCS7 de forma automática.
- Integridad: Aplica un HMAC (Hash-based Message Authentication Code) usando SHA256. Esto funciona como una firma digital; si un bit del texto cifrado cambia, el descifrado fallará automáticamente.
- IV Aleatorio: Genera un Vector de Inicialización seguro y diferente cada vez que llamas a la función de cifrar.

In [10]:
from cryptography.fernet import Fernet

# Fernet genera una clave segura automáticamente, codificada en base64
key = Fernet.generate_key()
f = Fernet(key)
msg = b"Transfer 100 euros"

# Ciframos el mismo mensaje dos veces exactas
ct1 = f.encrypt(msg)
ct2 = f.encrypt(msg)

print("Texto cifrado 1:", ct1)
print("Texto cifrado 2:", ct2)

# Desciframos el primer texto
pt = f.decrypt(ct1)
print("Texto plano descifrado:", pt)

Texto cifrado 1: b'gAAAAABq2fazaKwefi4iJVcnbOdWCs_gUBmEEI7SjFmcPic8uYpdggSqXciVNtv4Pn13HPRqvF9VZYy5UyVCg0wrHxMbzRmv1E1IE6pdUWe1QV5FECBCci8='
Texto cifrado 2: b'gAAAAABq2fazR4sNrGMHztMhBYAyV-2zuuwCe2gYolQQ-W7BBWX4zMuJF89UVfyK2LeTtnqskVvwkV1BH0JRzf9kj99dGenvPWBykGK96TuxKH_KKIssiYE='
Texto plano descifrado: b'Transfer 100 euros'


**a) Ejecute varias veces y compare textos cifrados**

Ambos textos tienen cifrado distinto, y cambia con cada nueva ejecucion del codigo.

**b) ¿El texto cifrado cambia en cada ejecución, aunque el mensaje y la clave sean los mismos? ¿Por qué?**

Cambia siempre, ya que la funcion encrypt genera un nuevo vector de inicializacion (IV) aleatorio de 128 bits cada vez que se invoca. el IV se concaneta con el texto cifrado final

**c) ¿Qué propiedades de seguridad ofrece Fernet frente al ejemplo de CBC de la Parte 1?**

Confidencialidad mediante un IV dinamico y aleatorio; Integridad y Autenticacion garantizando que el mensaje no sea modificado por un tercero. 

**d) ¿Qué mecanismos incorpora Fernet por defecto que en la Parte 1 tendrían que implementarse explícitamente?**

- generacion de un IV seguro y aleatorio por cada mensaje
- Padding estandarizado con PKCS7 en lugar de manual
- codigo de autenticacion de mensajes HMAC para asegurar integridad de los datos 
- sello de tiempo claro para saber cuando se genero el token (ver en ejercicio 4)

**e) ¿Qué primitivas criptográficas utiliza Fernet internamente y cómo las combina? ¿Es equivalente a usar un modo AEAD como AES-GCM?**

Utiliza AES-128 en modo CBC para la confidencialidad y HMAC-SHA256 para la autenticación. Se combinan usando el paradigma Encrypt-then-MAC, osea que primero cifra y luego calcula el hash HMAC sobre el texto cifrado resultante y el IV.

A nivel de objetivos de seguridad, es eqivalente a un modo AEAD ya que ambos usan cifrado autenticado. Sin embargo a nivel tecnico son diferentes. AES-GCM es un único algoritmo de cifrado en bloque que hace ambas cosas matemáticamente de forma nativa e integrada, mientras que Fernet es una construcción modular que junta dos algoritmos distintos (AES + HMAC) para lograr el mismo fin.

## Ejercicio 4

### Razonamiento

cuando Fernet genera un token cifrado, incrusta automáticamente un sello de tiempo (timestamp) que registra el momento exacto de su creación. El parámetro ttl (Time-To-Live) significa "Tiempo de vida" e indica una ventana de tiempo en segundos. Durante el descifrado, Fernet extrae el timestamp del token, revisa la hora actual del sistema y evalúa si la diferencia entre ambos tiempos es menor o igual al ttl permitido.

In [1]:
import time
from cryptography.fernet import Fernet, InvalidToken

key = Fernet.generate_key()
f = Fernet(key)
msg = b"Transfer 100 euros"

# Ciframos el mensaje
ct = f.encrypt(msg)
print("Token original generado.")

# 1. Prueba de descifrado exitoso (dentro del tiempo límite)
try:
    pt = f.decrypt(ct, ttl=60) # ttl de 60 segundos
    print("Descifrado normal exitoso:", pt)
except InvalidToken:
    print("Error en descifrado normal.")

# 2. Prueba forzando la expiración del token
time.sleep(1) # Esperamos 1 segundo
try:
    # Exigimos un ttl de 0 segundos, por lo que el token de 1 segundo de antigüedad fallará
    pt_expirado = f.decrypt(ct, ttl=0) 
except InvalidToken:
    print("Fallo de tiempo esperado: El token superó el TTL permitido.")

Token original generado.
Descifrado normal exitoso: b'Transfer 100 euros'
Fallo de tiempo esperado: El token superó el TTL permitido.


**¿Qué significa ttl?**

Significa "Time-to-Live" (Tiempo de vida). Es el tiempo máximo en segundos durante el cual el token se considera válido para ser descifrado.   

**¿Qué ocurre con un token demasiado antiguo?**

Al intentar descifrarlo, la función verifica que el tiempo transcurrido desde la creación del token supera el límite especificado en ttl y levanta una excepción de seguridad (InvalidToken), bloqueando la obtención del texto plano.   

**¿Y si el token ha sido modificado?**

El descifrado falla instantáneamente lanzando la misma excepción (InvalidToken). Antes de verificar el ttl o intentar descifrar, Fernet valida el sello criptográfico (HMAC). Si alguien alteró un solo byte (incluso si solo intentó modificar el timestamp para ganar más tiempo), la firma no coincide y el token es rechazado por pérdida de integridad.

## Ejercicio 5

### Razonamiento (AES-GCM (Galois/Counter Mode))

**AEAD (Authenticated Encryption with Associated Data)**: A diferencia de AES-CBC (que solo cifra) o Fernet (que combina AES-CBC con HMAC de forma separada), GCM entrelaza el cifrado y la autenticación matemáticamente en un solo paso utilizando multiplicaciones en campos de Galois. Garantiza confidencialidad e integridad de manera simultánea y nativa.

**Nonce (Number used ONCE)**: Es el equivalente al IV, pero su regla matemática es mucho más estricta. Nunca debe repetirse bajo la misma clave.

**AAD (Additional Authenticated Data)**: Permite añadir datos en texto claro (como cabeceras de red, direcciones o metadatos) que no se cifran, pero sí se autentican. Si un atacante altera el AAD, el descifrado fallará, garantizando que el contexto del mensaje es legítimo.

In [2]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

# Generación de clave y configuración
key = AESGCM.generate_key(bit_length=128)
aesgcm = AESGCM(key)
nonce = os.urandom(12)

msg = b"Transfer 100 euros"
aad = b"cabecera"

# Cifrado normal
ct = aesgcm.encrypt(nonce, msg, aad)
print("Texto cifrado original (hex):", ct.hex())

# Descifrado normal exitoso
pt = aesgcm.decrypt(nonce, ct, aad)
print("Descifrado original exitoso:", pt)

# Modificamos intencionalmente el primer byte del texto cifrado (operación XOR) simulando un ataque
ct_modificado = bytearray(ct)
ct_modificado[0] ^= 0xFF 
ct_modificado = bytes(ct_modificado)

print("\nIntentando descifrar texto modificado...")
try:
    aesgcm.decrypt(nonce, ct_modificado, aad)
except Exception as e:
    print("Fallo de seguridad esperado. El descifrado fue bloqueado. Error:", type(e).__name__)

Texto cifrado original (hex): 9bc590e532b396afca44bdde633730b4efd5f020cd1fc1f6f0d4d0e631680989a0f1
Descifrado original exitoso: b'Transfer 100 euros'

Intentando descifrar texto modificado...
Fallo de seguridad esperado. El descifrado fue bloqueado. Error: InvalidTag


**a) Modifique después un byte del texto cifrado o del AAD y compruebe qué ocurre al descifrar**

Al intentar descifrar el mensaje alterado, la librería lanza una excepción de criptografía (específicamente InvalidTag). Esto ocurre porque la etiqueta de autenticación generada durante el cifrado no coincide con la huella matemática del texto cifrado modificado, bloqueando el acceso al texto plano.

**b) ¿Qué propiedades de seguridad proporciona AES-GCM?**

Proporciona Confidencialidad (el mensaje queda ilegible) e Integridad/Autenticación (garantiza que el mensaje y el AAD no han sido alterados y provienen de quien posee la clave secreta).   

**c) ¿Por qué no debe reutilizarse un nonce con la misma clave?**

En la estructura matemática de GCM, reutilizar un nonce con la misma clave permite a un atacante calcular la clave de autenticación secundaria (hash key). Con esta clave deducida, el atacante puede falsificar etiquetas criptográficas e inyectar mensajes fraudulentos sin conocer la clave maestra de cifrado, destruyendo por completo la integridad del sistema.   

**c) Compare brevemente AES-GCM con CBC y con Fernet:**

Frente a CBC: CBC requiere implementar manualmente el relleno (padding) y no ofrece integridad por sí solo, siendo vulnerable a manipulaciones. GCM no necesita padding (funciona internamente como un cifrado de flujo) y resuelve la integridad nativamente.   

Frente a Fernet: Fernet es un formato de alto nivel preempaquetado y rígido (usa AES-128 en modo CBC junto a un HMAC-SHA256 tradicional). AES-GCM es una primitiva criptográfica pura, mucho más rápida computacionalmente gracias a que sus operaciones son paralelizables, y permite el uso de AAD (datos asociados no cifrados), una característica que Fernet no incluye.  